# Model Comparison: Baseline vs Range-Extended Encodings vs DeepReflecs

Compares the baseline histogram MLP against a validated 5-feature encoding (`x_rel`/`y_rel`/`vr_compensated`/`range_sc`/`rcs`, baseline's own Cartesian features plus `range_sc`) tested across four different reductions, and DeepReflecs, the point-set network (`deepreflecs.md`). All numbers are 6-fold split-sensitivity means, not single-split, an earlier draft compared single-split numbers, mistook the gap for a noise-floor artifact, and was wrong; see `MLP_MaxAD_Findings.md` for that history, including an even earlier polar-feature (`radial`/`azimuth_sc`) attempt that underperformed baseline outright and isn't included here. Every cell reads already-cached `split_sensitivity_summary.csv`/`*_val_metrics.json` files, no retraining. Full writeup: `model_comparison.md`.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MLP_DIR = PROJECT_ROOT / "results" / "mlp"
DEEPREFLECS_DIR = PROJECT_ROOT / "results" / "deepreflecs"
CLASSES = ["car", "large_vehicle", "two_wheeler", "pedestrian", "pedestrian_group"]

# (label, 6-fold split_search dir, per-fold metrics filename, point-level feature set, reduction)
METHODS = [
    ("baseline", MLP_DIR / "split_search", "mlp_val_metrics.json",
     ["rcs", "vr_compensated", "x_rel", "y_rel"],
     "16-bin equal-width histogram + doppler_spread unbinned"),
    ("histogram_5features", MLP_DIR / "split_search_histogram_5features", "mlp_val_metrics.json",
     ["x_rel", "y_rel", "vr_compensated", "range_sc", "rcs"],
     "16-bin equal-width histogram + doppler_spread unbinned"),
    ("quantile_bins_5features", MLP_DIR / "split_search_quantile_bins_5features", "mlp_val_metrics.json",
     ["x_rel", "y_rel", "vr_compensated", "range_sc", "rcs"],
     "16-bin quantile histogram + doppler_spread unbinned"),
    ("stat_descriptors_5features", MLP_DIR / "split_search_stat_descriptors_5features", "mlp_val_metrics.json",
     ["x_rel", "y_rel", "vr_compensated", "range_sc", "rcs"],
     "mean/median/std per feature + doppler_spread unbinned"),
    ("maxad_stats_5features", MLP_DIR / "split_search_maxad_stats_5features", "mlp_val_metrics.json",
     ["x_rel", "y_rel", "vr_compensated", "range_sc", "rcs"],
     "median/maxAD per feature + doppler_spread unbinned"),
    ("deepreflecs", DEEPREFLECS_DIR / "split_search", "deepreflecs_val_metrics.json",
     ["x_rel", "y_rel", "rcs", "vr_compensated", "range_sc"],
     "no reduction, raw point list, shared-weight point-set network"),
]
N_FOLDS = 6


def fold_per_class_f1_all(split_search_dir: Path, metrics_filename: str) -> pd.DataFrame:
    """Reads each fold_<n>/<metrics_filename>, returns a DataFrame of per-class F1, one column
    per fold (named by fold number, not left as duplicate 'f1' labels, so two such DataFrames
    can be subtracted safely without relying on positional alignment of matching duplicate
    labels). Computing macro F1 directly from these (rather than a split_sensitivity_summary.csv,
    which not every split-sensitivity script saves, e.g. split_sensitivity.py's own baseline run)
    keeps this notebook working off the one thing every method's cache is guaranteed to have."""
    return pd.concat(
        [pd.read_json(split_search_dir / f"fold_{i}" / metrics_filename, orient="index").loc[CLASSES]["f1"]
         .rename(i)
         for i in range(N_FOLDS)],
        axis=1,
    )

## Feature sets used

The only feature-set difference between baseline and everything else here is `range_sc`. All four MLP variants and DeepReflecs otherwise reduce the same Cartesian point-level quantities.

In [2]:
feature_set_df = pd.DataFrame(
    [{"method": name, "point_level_features": ", ".join(features), "reduction": reduction}
     for name, _, _, features, reduction in METHODS]
).set_index("method")
feature_set_df

,point_level_features,reduction
method,,
baseline,"rcs, vr_compensated, x_rel, y_rel",16-bin equal-width histogram + doppler_spread ...
histogram_5features,"x_rel, y_rel, vr_compensated, range_sc, rcs",16-bin equal-width histogram + doppler_spread ...
quantile_bins_5features,"x_rel, y_rel, vr_compensated, range_sc, rcs",16-bin quantile histogram + doppler_spread unb...
stat_descriptors_5features,"x_rel, y_rel, vr_compensated, range_sc, rcs",mean/median/std per feature + doppler_spread u...
maxad_stats_5features,"x_rel, y_rel, vr_compensated, range_sc, rcs",median/maxAD per feature + doppler_spread unbi...
deepreflecs,"x_rel, y_rel, rcs, vr_compensated, range_sc","no reduction, raw point list, shared-weight po..."


## Results: 6-fold split-sensitivity (validated)

In [3]:
per_class_by_fold = {
    name: fold_per_class_f1_all(split_search_dir, metrics_filename)
    for name, split_search_dir, metrics_filename, _, _ in METHODS
}
fold_macro = {name: df.mean(axis=0).to_numpy() for name, df in per_class_by_fold.items()}
baseline_vals = fold_macro["baseline"]

rows = []
for name, vals in fold_macro.items():
    if name == "baseline":
        rows.append({"method": name, "mean_macro_f1": vals.mean(), "mean_delta": np.nan,
                     "delta_std": np.nan, "wins": ""})
    else:
        deltas = vals - baseline_vals
        rows.append({"method": name, "mean_macro_f1": vals.mean(), "mean_delta": deltas.mean(),
                     "delta_std": deltas.std(ddof=1), "wins": f"{(deltas > 0).sum()}/{N_FOLDS}"})

results_df = pd.DataFrame(rows).set_index("method").round(4)
results_df

,mean_macro_f1,mean_delta,delta_std,wins
method,,,,
baseline,0.6915,NaN,NaN,
histogram_5features,0.7141,0.0226,0.0071,6/6
quantile_bins_5features,0.7229,0.0314,0.0090,6/6
stat_descriptors_5features,0.7253,0.0338,0.0069,6/6
maxad_stats_5features,0.7230,0.0315,0.0081,6/6
deepreflecs,0.7350,0.0435,0.0092,6/6


## Per-class F1 (6-fold mean)

In [4]:
per_class_mean = pd.DataFrame({name: df.mean(axis=1) for name, df in per_class_by_fold.items()}).round(3)
per_class_std = pd.DataFrame({name: df.std(axis=1, ddof=1) for name, df in per_class_by_fold.items()}).round(3)
per_class_df = per_class_mean.astype(str) + " ± " + per_class_std.astype(str)
per_class_df

,baseline,histogram_5features,quantile_bins_5features,stat_descriptors_5features,maxad_stats_5features,deepreflecs
car,0.863 ± 0.02,0.866 ± 0.017,0.865 ± 0.02,0.868 ± 0.02,0.864 ± 0.022,0.877 ± 0.016
large_vehicle,0.719 ± 0.065,0.713 ± 0.055,0.696 ± 0.037,0.719 ± 0.049,0.714 ± 0.045,0.735 ± 0.049
two_wheeler,0.571 ± 0.133,0.614 ± 0.106,0.644 ± 0.097,0.647 ± 0.092,0.635 ± 0.091,0.65 ± 0.097
pedestrian,0.674 ± 0.03,0.684 ± 0.03,0.694 ± 0.036,0.69 ± 0.028,0.694 ± 0.029,0.703 ± 0.03
pedestrian_group,0.63 ± 0.045,0.693 ± 0.028,0.716 ± 0.038,0.702 ± 0.034,0.708 ± 0.037,0.71 ± 0.042


## Per-class verdict: real or noise

The table above mixes two different effects (feature choice: any 5-feature variant vs baseline; architecture: DeepReflecs vs the best 5-feature MLP variant), so "best per class" isn't the same question as "is that difference real." Splitting them: wins across the 6 folds, and whether the mean delta clearly exceeds its own fold-to-fold std.

In [5]:
def per_class_verdict(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    """a/b: per_class_by_fold entries (rows=class, cols=fold). Returns wins/mean/std of (a-b)
    per class, one row per class, in CLASSES order."""
    deltas = a - b
    return pd.DataFrame({
        "wins": (deltas > 0).sum(axis=1).astype(str) + f"/{N_FOLDS}",
        "mean_delta": deltas.mean(axis=1),
        "delta_std": deltas.std(axis=1, ddof=1),
    }).round(4)

deepreflecs_vs_baseline = per_class_verdict(per_class_by_fold["deepreflecs"], per_class_by_fold["baseline"])
deepreflecs_vs_best_mlp = per_class_verdict(per_class_by_fold["deepreflecs"], per_class_by_fold["quantile_bins_5features"])

verdict_df = pd.concat(
    {"vs_baseline": deepreflecs_vs_baseline, "vs_best_mlp_variant (architecture only)": deepreflecs_vs_best_mlp},
    axis=1,
)
verdict_df

vs_baseline                       \
                        wins mean_delta delta_std   
car                      6/6     0.0135    0.0064   
large_vehicle            4/6     0.0161    0.0266   
two_wheeler              6/6     0.0790    0.0429   
pedestrian               6/6     0.0285    0.0140   
pedestrian_group         6/6     0.0806    0.0171   

                 vs_best_mlp_variant (architecture only)                       
                                                    wins mean_delta delta_std  
car                                                  6/6     0.0120    0.0060  
large_vehicle                                        6/6     0.0388    0.0161  
two_wheeler                                          6/6     0.0064    0.0030  
pedestrian                                           6/6     0.0091    0.0097  
pedestrian_group                                     2/6    -0.0055    0.0104

DeepReflecs' advantage over baseline is real for 4 of 5 classes (`large_vehicle` is noise both ways here too). But once feature choice is separated out (the `vs_best_mlp_variant` column), the architecture effect on its own is only clearly real for `car`, negligible for `two_wheeler`/`pedestrian`, and slightly reversed for `pedestrian_group` (DeepReflecs loses to `quantile_bins_5features` on that class in 4 of 6 folds). Most of the per-class story belongs to `range_sc`, not the network. See `model_comparison.md` and `deepreflecs.md`'s "Per-class check across folds" section for the full writeup.

## Comparison

Encoding scheme genuinely doesn't matter: the four 5-feature MLP variants land within 0.011 macro F1 of each other (0.714 to 0.725), tighter than any one of their own fold-to-fold spreads. Equal-width bins, quantile bins, raw mean/median/std, raw median/maxAD, computed from the same 5 features, all end up in the same place.

What separates baseline from all four is exactly one feature, `range_sc`. That's a validated effect (+0.02 to +0.03 macro F1, confirmed across all 6 folds, tight delta std 0.007-0.009), not noise, reversing what an earlier draft of this notebook concluded from single-split numbers alone.

DeepReflecs adds a further, smaller effect on top: paired directly against `quantile_bins_5features` (identical feature set, same 6 folds), mean +0.012 macro F1, wins 6/6, delta std 0.004. A real, distinct architecture effect, additive to the feature-set effect rather than a separate confound.

Net: DeepReflecs' total 6-fold advantage over baseline (+0.044) decomposes almost exactly into feature choice (+0.031, using `quantile_bins_5features` as the representative encoding) plus architecture (+0.012). Both pieces are validated. See `model_comparison.md` for the full writeup and `MLP_MaxAD_Findings.md` for how the polar-feature dead end led here.